# Critical points at $e=0$ — numerics only

For $H=\mathbb T,\mathbb O,\mathbb I$: find the critical points of the reduced functional (11)
at eccentricity $e=0$, in a fundamental domain of the two angles, and check nondegeneracy.

At $e=0$ the functional depends only on the orbit-plane normal $\nu\in S^2$:
$$\Psi(\nu)=\tfrac12\int_0^{2\pi}S(\hat u(\theta))\,d\theta,\qquad
S(\hat u)=\sum_{L\ne\mathrm{Id}}\frac{1}{\|\hat u-L\hat u\|},$$
$\hat u$ running over the great circle $\nu^\perp$.

In [1]:
using LinearAlgebra, Printf

## 1. The three groups

In [2]:
## rotation of angle α about the axis n
function rot(n, α)
    n = normalize(float.(collect(n)))
    K = [0 -n[3] n[2]; n[3] 0 -n[1]; -n[2] n[1] 0]
    Matrix(1.0I,3,3) + sin(α)*K + (1-cos(α))*K^2
end

function closure(gens)
    G = [Matrix(1.0I,3,3)]
    changed = true
    while changed
        changed = false
        for X in copy(G), s in gens
            Y = X*s
            any(M -> norm(M-Y) < 1e-9, G) || (push!(G, Y); changed = true)
        end
    end
    G
end

const φg = (1 + sqrt(5))/2                     # golden ratio
const MINUS = -Matrix(1.0I,3,3)

## NOTE: these generators are written as axis-angle rotations instead of the matrices A, B of the
## paper's appendix, but they generate THE SAME GROUPS: the closures below are the identical sets of
## 12, 24 and 60 matrices as the closures of the paper's generators (verified element by element),
## in the same frame — not merely isomorphic. All coordinates below are therefore in the paper's frame.
Tet = closure([rot([1,0,0], pi),   rot([1,1,1], 2pi/3)])       # T,  order 12
Oct = closure([rot([0,0,1], pi/2), rot([1,1,1], 2pi/3)])       # O,  order 24
Ico = closure([rot([0,0,1], pi),   rot([0,1,φg], 2pi/5)])      # I,  order 60

## Γ = N_{O(3)}(H):  O_h for T and O,  I_h for I
Gam_cub = closure([rot([0,0,1], pi/2), rot([1,1,1], 2pi/3), MINUS])
Gam_ico = closure([rot([0,0,1], pi),   rot([0,1,φg], 2pi/5), MINUS])

@printf("|T| = %d   |O| = %d   |I| = %d      |Γ_cub| = %d   |Γ_ico| = %d\n",
        length(Tet), length(Oct), length(Ico), length(Gam_cub), length(Gam_ico))

|T| = 12   |O| = 24   |I| = 60      |Γ_cub| = 48   |Γ_ico| = 120


## 2. $\Psi(\nu)$

For a rotation of angle $\alpha$ about the unit axis $n$,
$\|\hat u-L\hat u\| = \sqrt{3-\mathrm{tr}\,L}\;\sqrt{1-(\hat u\cdot n)^2}$, and along the great
circle $\hat u\cdot n = \rho\cos(\theta-\theta_0)$ with $\rho^2=1-(\nu\cdot n)^2$. Since
$\int_0^{2\pi}\frac{d\theta}{\sqrt{1-\rho^2\cos^2\theta}}=4K(\rho)=\frac{2\pi}{M(1,|\nu\cdot n|)}$
($M$ = arithmetic–geometric mean),
$$\Psi(\nu)=\sum_{L\ne\mathrm{Id}}\frac{\pi}{\sqrt{3-\mathrm{tr}\,L}\;\;M\bigl(1,|\nu\cdot n_L|\bigr)} .$$
$\Psi=+\infty$ exactly when $\nu\perp$ some axis (collision). Checked against direct quadrature below.

In [3]:
## (axis, sqrt(3-tr L)) for every non-identity element
axes_weights(G) = [(vec(nullspace(L - Matrix(1.0I,3,3))), sqrt(max(3 - tr(L), 0.0)))
                   for L in G if norm(L - Matrix(1.0I,3,3)) > 1e-9]

function agm(x, y)
    for _ in 1:30; x, y = (x+y)/2, sqrt(x*y); end
    x
end

Psi(nu, AW) = sum(pi/(c * agm(1.0, abs(dot(nu, n)))) for (n, c) in AW)

## direct definition, for checking only
Sdir(u, AW) = sum(1/(c*sqrt(max(1 - dot(u,n)^2, 0.0))) for (n,c) in AW)
function Psi_quad(nu, AW; N = 20_000)
    a = abs(nu[1]) < 0.9 ? [1.0,0,0] : [0.0,1,0]
    e1 = normalize(a - dot(a,nu)*nu); e2 = cross(nu, e1)
    0.5 * sum(Sdir(cos(2pi*k/N)*e1 + sin(2pi*k/N)*e2, AW) for k in 0:N-1) * (2pi/N)
end

let nu = normalize([0.31, 0.57, 0.76])
    for (nm, G) in (("T", Tet), ("O", Oct), ("I", Ico))
        AW = axes_weights(G)
        @printf("check %s :  Ψ = %.12f   quadrature = %.12f   rel = %.1e\n",
                nm, Psi(nu,AW), Psi_quad(nu,AW), abs(Psi(nu,AW)-Psi_quad(nu,AW))/Psi(nu,AW))
    end
end

check T :  Ψ = 30.323450753040   quadrature = 30.323450753040   rel = 1.9e-15
check O :  Ψ = 62.957109564510   quadrature = 62.957109564510   rel = 1.4e-15
check I :  Ψ = 170.271568814463   quadrature = 170.271568814463   rel = 4.2e-15


## 3. The fundamental domain of the two angles

Take a generic $p_0\in S^2$. The set
$$D=\{\nu\in S^2:\ \nu\cdot p_0 \ \ge\ \nu\cdot(\gamma p_0)\ \ \forall\gamma\in\Gamma\}$$
(the Dirichlet cell of $p_0$) is a fundamental domain: every $\nu$ has exactly one image in $D$,
except on $\partial D$, which is where the isotropy sits. Here $\Gamma$ is generated by reflections,
so $D$ is a spherical triangle of area $4\pi/|\Gamma|$, and its three vertices are rotation axes of
$\Gamma$. `toFD` moves any $\nu$ into $D$.

In [4]:
fd_targets(Gam, p0) = hcat([M*p0 for M in Gam]...)          # the points γp₀

inFD(nu, p0, P) = dot(nu, p0) ≥ maximum(P' * nu) - 1e-12

function toFD(nu, Gam, p0)
    best = Gam[1]; bv = -Inf
    for M in Gam
        v = dot(nu, M*p0); v > bv && (bv = v; best = M)
    end
    best' * nu
end

## p₀ generic (on no mirror of Γ)
const P0_CUB = normalize([3.0, 2.0, 1.0])       # its cell is exactly {ν₁ ≥ ν₂ ≥ ν₃ ≥ 0}
const P0_ICO = normalize([0.17, 0.41, 1.0])

nuof(φ, θ) = [sin(θ)*cos(φ), sin(θ)*sin(φ), cos(θ)]
function angs(nu)
    φ = mod(atan(nu[2], nu[1]), 2pi)
    (φ > 2pi - 1e-9 ? 0.0 : φ, acos(clamp(nu[3], -1, 1)))
end

## the vertices of D: the rotation axes of Γ that lie in D
function fd_vertices(Gam, p0)
    P = fd_targets(Gam, p0)
    V = Vector{Float64}[]
    rotations = [L for L in Gam if det(L) > 0.5 && norm(L - Matrix(1.0I,3,3)) > 1e-9]
    for L in rotations
        n = vec(nullspace(L - Matrix(1.0I,3,3)))
        for s in (n, -n)
            inFD(s, p0, P) && !any(w -> norm(w - s) < 1e-8, V) && push!(V, s)
        end
    end
    V
end

for (nm, Gam, p0) in (("T and O  (Γ = O_h, |Γ| = 48)", Gam_cub, P0_CUB),
                      ("I        (Γ = I_h, |Γ| = 120)", Gam_ico, P0_ICO))
    println("\nfundamental domain for ", nm, ",   area = 4π/", length(Gam), " = ",
            round(4pi/length(Gam), digits = 5))
    for v in fd_vertices(Gam, p0)
        φ, θ = angs(v)
        @printf("   vertex ν = (%+.6f, %+.6f, %+.6f)    φ = %8.4f°   ϑ = %8.4f°\n",
                v..., rad2deg(φ), rad2deg(θ))
    end
end


fundamental domain for T and O  (Γ = O_h, |Γ| = 48),   area = 4π/48 = 0.2618
   vertex ν = (+0.577350, +0.577350, +0.577350)    φ =  45.0000°   ϑ =  54.7356°
   vertex ν = (+1.000000, -0.000000, -0.000000)    φ =   0.0000°   ϑ =  90.0000°
   vertex ν = (+0.707107, +0.707107, -0.000000)    φ =  45.0000°   ϑ =  90.0000°

fundamental domain for I        (Γ = I_h, |Γ| = 120),   area = 4π/120 = 0.10472
   vertex ν = (-0.000000, +0.525731, +0.850651)    φ =  90.0000°   ϑ =  31.7175°
   vertex ν = (+0.356822, -0.000000, +0.934172)    φ =   0.0000°   ϑ =  20.9052°
   vertex ν = (+0.500000, +0.309017, +0.809017)    φ =  31.7175°   ϑ =  36.0000°


## 4. The search

Grid over $D$ → local minima of $\Psi$ → Newton on $\nabla\Psi$ in the chart $(\varphi,\vartheta)$
→ map back into $D$ → keep those with $\|\nabla\Psi\|\approx0$ → remove duplicates.
Reported: $\Psi$, the two eigenvalues of the spherical Hessian $\nabla^2_{(\psi_1,\psi_2)}\Psi$, and
$|\hat S_2|$, the second Fourier coefficient of $\theta\mapsto S(\hat u(\theta))$, whose $\pm\frac\pi2|\hat S_2|$
are the eigenvalues of the eccentricity block. The $4\times4$ Hessian is nondegenerate iff both are.

The search starts from grid minima, so it returns the **minimum of each chamber** — the ones the
chamber argument of Section 7 predicts. Saddles interior to a chamber, if any, are not sought.
The counts below are unchanged on grids up to $4200\times2100$.

In [5]:
function make_derivs(AW)
    f(φ, θ) = Psi(nuof(φ, θ), AW)
    h = 1e-5; H = 1e-4
    grad(φ, θ) = [(f(φ+h,θ) - f(φ-h,θ))/(2h), (f(φ,θ+h) - f(φ,θ-h))/(2h)]
    function hess(φ, θ)
        m = (f(φ+H,θ+H) - f(φ+H,θ-H) - f(φ-H,θ+H) + f(φ-H,θ-H))/(4H^2)
        [ (f(φ+H,θ)-2f(φ,θ)+f(φ-H,θ))/H^2  m ;  m  (f(φ,θ+H)-2f(φ,θ)+f(φ,θ-H))/H^2 ]
    end
    f, grad, hess
end

## Hessian of Ψ on the sphere, in an orthonormal tangent frame at ν
function hess_sphere(nu, AW; h = 1e-4)
    t1 = normalize(abs(nu[1]) < 0.9 ? cross(nu, [1.0,0,0]) : cross(nu, [0.0,1,0]))
    t2 = cross(nu, t1)
    F(s, t) = Psi(normalize(nu + s*t1 + t*t2), AW)
    m = (F(h,h) - F(h,-h) - F(-h,h) + F(-h,-h))/(4h^2)
    [ (F(h,0)-2F(0,0)+F(-h,0))/h^2  m ;  m  (F(0,h)-2F(0,0)+F(0,-h))/h^2 ]
end

## |Ŝ₂| along the great circle of normal ν
function S2(nu, AW; N = 8192)
    a = abs(nu[1]) < 0.9 ? [1.0,0,0] : [0.0,1,0]
    e1 = normalize(a - dot(a,nu)*nu); e2 = cross(nu, e1)
    ca = 0.0; cb = 0.0
    for k in 0:N-1
        θ = 2pi*k/N; s = Sdir(cos(θ)*e1 + sin(θ)*e2, AW)
        ca += s*cos(2θ); cb += s*sin(2θ)
    end
    hypot(2ca/N, 2cb/N)
end

function critical_points(G, Gam, p0; nφ = 1440, nθ = 720)
    AW = axes_weights(G)
    P  = fd_targets(Gam, p0)
    f, grad, hess = make_derivs(AW)

    ## 1. grid over D
    pts = Tuple{Int,Int}[]; val = Dict{Tuple{Int,Int},Float64}()
    for i in 0:nφ-1, j in 1:nθ-1
        φ = 2pi*i/nφ; θ = pi*j/nθ
        if inFD(nuof(φ, θ), p0, P)
            val[(i,j)] = f(φ, θ); push!(pts, (i,j))
        end
    end

    ## 2. local minima of Ψ on that grid
    cand = Tuple{Float64,Float64}[]
    for (i,j) in pts
        v = val[(i,j)]
        ok = true
        for di in -1:1, dj in -1:1
            (di == 0 && dj == 0) && continue
            get(val, (mod(i+di, nφ), j+dj), Inf) < v && (ok = false; break)
        end
        ok && push!(cand, (2pi*i/nφ, pi*j/nθ))
    end

    ## 3. Newton, fold back into D, keep genuine critical points
    out = Vector{Float64}[]
    for (φ0, θ0) in cand
        p = [φ0, θ0]
        for _ in 1:200
            d = hess(p...) \ grad(p...)
            nd = norm(d); nd > 0.05 && (d *= 0.05/nd)
            p -= d
            (!isfinite(norm(p)) || p[2] < 1e-3 || p[2] > pi-1e-3) && break
            nd < 1e-13 && break
        end
        (!isfinite(norm(p)) || p[2] < 1e-3 || p[2] > pi-1e-3) && continue
        nu = toFD(nuof(p...), Gam, p0)
        q  = collect(angs(nu))
        (isfinite(f(q...)) && norm(grad(q...)) < 1e-5) || continue
        any(w -> norm(w - nu) < 1e-6, out) || push!(out, nu)
    end
    sort!(out, by = nu -> Psi(nu, AW))
    out, AW
end

critical_points (generic function with 1 method)

## 5. Results

In [6]:
for (nm, G, Gam, p0) in (("T  (tetrahedral, n = 12)",  Tet, Gam_cub, P0_CUB),
                         ("O  (octahedral,  n = 24)",  Oct, Gam_cub, P0_CUB),
                         ("I  (icosahedral, n = 60)",  Ico, Gam_ico, P0_ICO))
    crit, AW = critical_points(G, Gam, p0)
    println("\n", "="^100)
    println(nm, " :  ", length(crit), " critical orientation(s) in the fundamental domain")
    println("="^100)
    _, grad, hess = make_derivs(AW)
    for (k, nu) in enumerate(crit)
        φ, θ = angs(nu); ev = eigvals(hess_sphere(nu, AW)); s2 = S2(nu, AW)
        @printf("\n #%d   ν = (%+.9f, %+.9f, %+.9f)\n", k, nu...)
        @printf("      φ = %.6f°   ϑ = %.6f°        (paper: ψ₁ = %.6f°, ψ₂ = %.6f°)\n",
                rad2deg(φ), rad2deg(θ), rad2deg(atan(-nu[2], nu[3])), rad2deg(-asin(nu[1])))
        @printf("      Ψ  = %.10f          |∇Ψ| = %.1e\n", Psi(nu, AW), norm(grad(φ, θ)))
        @printf("      orientation block eig = (%.4f, %.4f)   det = %.4f\n", ev..., ev[1]*ev[2])
        @printf("      |Ŝ₂| = %.8f  → eccentricity block eig = ±%.8f\n", s2, (pi/2)*s2)
        @printf("      nondegenerate 4×4 Hessian : %s\n",
                (abs(ev[1]*ev[2]) > 1e-8 && s2 > 1e-8) ? "YES" : "NO   (Ŝ₂ = 0: extra symmetry of the orbit plane)")
    end
end


T  (tetrahedral, n = 12) :  2 critical orientation(s) in the fundamental domain

 #1   ν = (+0.957714422, +0.203449116, +0.203449116)
      φ = 11.993167°   ϑ = 78.261273°        (paper: ψ₁ = -45.000000°, ψ₂ = -73.278465°)
      Ψ  = 27.1650885744          |∇Ψ| = 1.8e-10
      orientation block eig = (45.6161, 56.1895)   det = 2563.1409
      |Ŝ₂| = 0.30929939  → eccentricity block eig = ±0.48584634
      nondegenerate 4×4 Hessian : YES

 #2   ν = (+0.577350269, +0.577350269, +0.577350269)
      φ = 45.000000°   ϑ = 54.735610°        (paper: ψ₁ = -45.000000°, ψ₂ = -35.264390°)
      Ψ  = 27.2333076172          |∇Ψ| = 0.0e+00
      orientation block eig = (42.2121, 42.2121)   det = 1781.8654
      |Ŝ₂| = 0.00000000  → eccentricity block eig = ±0.00000000
      nondegenerate 4×4 Hessian : NO   (Ŝ₂ = 0: extra symmetry of the orbit plane)

O  (octahedral,  n = 24) :  2 critical orientation(s) in the fundamental domain

 #1   ν = (+0.702710120, +0.572310609, +0.422680795)
      φ = 39.1605